# Hybrid Persistence + CNN-LSTM model



1.   Load the existing data/models
2.   Generate 2024 validation predictions from Persistence and CNN-LSTM.
3. Test ensemble weights from 0–100%.
4. Choose the weight using validation data only.
5. Freeze that weight.
6. Generate 2025 test predictions.
7. Calculate MAE, RMSE and R².
8. Compare Persistence, CNN-LSTM, Hybrid


No retraining


`Hybrid = w(Persistence) + (1−w) (CNN-LSTM)`

### Importing

In [ ]:
# importing

import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import load_model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Tensorflow version:", tf.__version__)

Tensorflow version: 2.20.0


### Load data and trained models

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load dataset

PROJECT_DIR = "/content/drive/MyDrive/SIH_Rainfall_Nowcasting"

DATA_PATH = "/content/drive/MyDrive/IMERG/chennai_imerg_30min_2021_2025.csv"

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [ ]:
# Recreate the train/val/test split
df["split"] = np.where(
    df["timestamp"] < "2024-01-01", "train",
    np.where(
        df["timestamp"] < "2025-01-01",
        "validation",
        "test"
    )
)

In [ ]:
# Load CNN-LSTM
CNN_LSTM_MODEL_PATH = f"{PROJECT_DIR}/models/CNN_LSTM/best_cnn_lstm.keras"

best_cnn_lstm_model = load_model(CNN_LSTM_MODEL_PATH)

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 16 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
# Print our values

print("Dataset shape:", df.shape)
print("CNN-LSTM loaded:", os.path.exists(CNN_LSTM_MODEL_PATH))
print("Validation observations:", (df["split"] == "validation").sum())
print("Test observations:", (df["split"] == "test").sum())

Dataset shape: (87648, 3)
CNN-LSTM loaded: True
Validation observations: 17568
Test observations: 17520


### Recreating preprocessing for hybrid experiment

In [ ]:
from sklearn.preprocessing import MinMaxScaler

LOOKBACK = 48
HORIZON = 1

# Split the data
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

In [ ]:
# Fit scaler ONLY on training data
scaler = MinMaxScaler()
scaler.fit(train_df[["rainfall_mm_hr"]].values)

MinMaxScaler()

In [ ]:
# Create sequences
def create_sequences(data, lookback=48, horizon=1):
    rainfall = data["rainfall_mm_hr"].values

    X = []
    y = []

    for i in range(lookback, len(rainfall) - horizon + 1):
        X.append(rainfall[i-lookback:i])
        y.append(rainfall[i+horizon-1])

    return np.array(X), np.array(y)

X_val, y_val = create_sequences(
    val_df,
    LOOKBACK,
    HORIZON
)

In [ ]:
# Scale validation inputs
X_val_scaled = scaler.transform(
    X_val.reshape(-1, 1)
).reshape(X_val.shape)

In [ ]:
# Add feature dimension for CNN-LSTM
X_val_lstm = X_val_scaled.reshape(
    X_val_scaled.shape[0],
    X_val_scaled.shape[1],
    1
)

In [ ]:
print("Validation input shape:", X_val_lstm.shape)
print("Validation target shape:", y_val.shape)

Validation input shape: (17520, 48, 1)
Validation target shape: (17520,)


### Now generate 2024 validation predictions

1. Persistence

In [ ]:
persistence_val_pred_scaled = X_val_scaled[:, -1]

persistence_val_predictions = scaler.inverse_transform(
    persistence_val_pred_scaled.reshape(-1, 1)
).flatten()

2. CNN-LSTM

In [ ]:
cnn_lstm_val_pred_scaled = best_cnn_lstm_model.predict(
    X_val_lstm,
    verbose=0
).flatten()

cnn_lstm_val_predictions = scaler.inverse_transform(
    cnn_lstm_val_pred_scaled.reshape(-1, 1)
).flatten()

3. Check sizes

In [ ]:
print("Validation observations :", len(y_val))
print("Persistence predictions  :", len(persistence_val_predictions))
print("CNN-LSTM predictions     :", len(cnn_lstm_val_predictions))

Validation observations : 17520
Persistence predictions  : 17520
CNN-LSTM predictions     : 17520


## Now finding best weights
We'll test weights from 0% to 100% Persistence in 5% steps. For each weight, we'll calculate MAE, RMSE, and R².

In [ ]:
weights = np.arange(0, 1.01, 0.05)

hybrid_results = []

for w in weights:

    hybrid_predictions = (
        w * persistence_val_predictions
        + (1 - w) * cnn_lstm_val_predictions
    )

    mae = mean_absolute_error(y_val, hybrid_predictions)
    rmse = np.sqrt(mean_squared_error(y_val, hybrid_predictions))
    r2 = r2_score(y_val, hybrid_predictions)

    hybrid_results.append({
        "Persistence weight": w,
        "CNN-LSTM weight": 1 - w,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

hybrid_results = pd.DataFrame(hybrid_results)

In [ ]:
display(hybrid_results.round(4))

,Persistence weight,CNN-LSTM weight,MAE,RMSE,R²
0,0.00,1.00,0.1150,0.5743,0.6660
1,0.05,0.95,0.1140,0.5741,0.6661
2,0.10,0.90,0.1129,0.5743,0.6659
3,0.15,0.85,0.1120,0.5748,0.6654
4,0.20,0.80,0.1110,0.5755,0.6645
5,0.25,0.75,0.1101,0.5765,0.6633
6,0.30,0.70,0.1091,0.5779,0.6618
7,0.35,0.65,0.1083,0.5795,0.6599
8,0.40,0.60,0.1075,0.5814,0.6577
9,0.45,0.55,0.1067,0.5835,0.6551
